In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import io

class ImageNetParquetDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_bytes = row["image"]["bytes"]
        label = row["label"]

        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("/home/shared/data/imagenet/validation-*.parquet"))
print(files)  # sanity check

df = pd.read_parquet(files)

In [ ]:

print(len(df))


In [ ]:
import numpy as np

SEED = 42
TEST_SAMPLES_PER_CLASS = 32

# Shuffle within each class, then split
test_df = (
    df.groupby("label", group_keys=False)
      .apply(lambda x: x.sample(n=TEST_SAMPLES_PER_CLASS, random_state=SEED))
)

train_df = df.drop(test_df.index)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


In [ ]:
test_df["label"].value_counts()


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import io

class ImageNetParquetDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_bytes = row["image"]["bytes"]
        label = row["label"]

        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


In [ ]:
import torch
from torchvision import models

weights = models.ResNet50_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
preprocess

In [ ]:
dataset = ImageNetParquetDataset(
    df,
    transform=preprocess
)

In [ ]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

weights = models.ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights)

# Remove classification head
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()


In [ ]:
from PIL import Image
import io
import matplotlib.pyplot as plt

# Get the image bytes
img_bytes = df.loc[10, 'image']['bytes']

# Convert bytes → PIL Image
img = Image.open(io.BytesIO(img_bytes))

# Plot
plt.imshow(img)
plt.axis("off")
plt.show()


---

In [ ]:
train_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/train.pt"))
val_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/validation.pt"))
test_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/test.pt"))

In [ ]:
def plot_images(imgs):
    n = len(imgs)
    cols = 5
    rows = int(np.ceil(n / cols))
    
    plt.figure(figsize=(cols * 3, rows * 3))
    
    for i, img in enumerate(imgs):
        plt.subplot(rows, cols, i + 1)
        
        # Handle grayscale vs RGB
        if img.ndim == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)
        
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
len(val_dataset)

In [ ]:
plot_images(train_dataset[random.randint(0, 70)][0])

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import random    

class SingleDigitPadded(Dataset):
    def __init__(self, mnist_dataset):
        self.mnist = mnist_dataset

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        img, label = self.mnist[idx]

        # Decide randomly whether the digit is on the left or right
        if random.random() < 0.5:
            # digit on left
            empty = torch.zeros_like(img)
            image = torch.cat((img, empty), dim=2)  # 28x56
            position = "left"
        else:
            # digit on right
            empty = torch.zeros_like(img)
            image = torch.cat((empty, img), dim=2)
            position = "right"

        return image, label, position
    

class TwoDigitOpposite(Dataset):
    def __init__(self, padded_dataset):
        """
        padded_dataset: your SingleDigitPadded dataset
        """
        self.images = padded_dataset["images"]
        self.positions = padded_dataset["positions"]
        self.labels = padded_dataset["labels"]

        # Precompute indices by position for faster sampling
        self.left_indices = [i for i in range(len(self.images)) if self.positions[i] == "left"]
        self.right_indices = [i for i in range(len(self.images)) if self.positions[i] == "right"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # First sample (fixed)
        img1, label1, pos1 = self.images[idx], self.labels[idx], self.positions[idx]

        # Determine opposite position
        opposite_pos = "right" if pos1 == "left" else "left"
        candidate_indices = self.right_indices if opposite_pos == "right" else self.left_indices

        # Randomly select second sample with opposite position
        random.seed(42)
        idx2 = random.choice(candidate_indices)
        img2, label2 = self.images[idx2], self.labels[idx2]

        # Combine images by adding (keep same shape)
        two_digit_img = img1 + img2
        two_digit_label = (label1, label2) if pos1 == "left" else (label2, label1)

        return two_digit_img, two_digit_label

    
def create_mnist1():
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    train_dataset = datasets.MNIST(
        root="/home/shared/data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="/home/shared/data",
        train=False,
        download=True,
        transform=transform
    )


    singleDigit_train = SingleDigitPadded(train_dataset)
    singleDigit_test = SingleDigitPadded(test_dataset)

    train_loader = DataLoader(singleDigit_train, batch_size=len(singleDigit_train))  
    test_loader = DataLoader(singleDigit_test, batch_size=len(singleDigit_test)) 

    for images, labels, positions in train_loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels,
            'positions': positions  
        }, '/home/shared/data/MNIST2/train.pt')
        break  # only one batch needed

    for images, labels, positions in test_loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels,
            'positions': positions
        }, '/home/shared/data/MNIST2/test.pt')
        break  # only one batch needed


def create_mnist2():
    singleDigit = torch.load('/home/shared/data/MNIST1/test.pt')
    
    two_digit_dataset = TwoDigitOpposite(singleDigit)

    # Save the dataset
    loader = DataLoader(two_digit_dataset, batch_size=len(two_digit_dataset))

    for images, labels in loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels[0] * 10 + labels[1],
        }, '/home/shared/data/MNIST2/test.pt')
        break

In [ ]:
create_mnist2()

In [ ]:
import torch
singleDigit = torch.load('/home/shared/data/MNIST2/test.pt')

In [ ]:
import matplotlib.pyplot as plt
import random
i = random.randint(0, 2000)
plt.imshow(singleDigit['images'][i].permute(1, 2, 0), cmap='gray')
plt.title(singleDigit['labels'][i])
plt.axis('off')
plt.show()

# Analysis

In [7]:
import numpy as np
from torch import nn
import torch
import torch.nn.functional as F

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class SlotAttention(nn.Module):
    def __init__(self, num_slots, dim, iters = 3, eps = 1e-8, hidden_dim = 128):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.eps = eps
        self.scale = dim ** -0.5

        self.slots_mu = nn.Parameter(torch.randn(1, 1, dim))
        self.slots_sigma = nn.Parameter(torch.rand(1, 1, dim))

        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, dim)
        self.to_v = nn.Linear(dim, dim)

        self.gru = nn.GRUCell(dim, dim)

        hidden_dim = max(dim, hidden_dim)

        self.fc1 = nn.Linear(dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, dim)

        self.norm_input  = nn.LayerNorm(dim)
        self.norm_slots  = nn.LayerNorm(dim)
        self.norm_pre_ff = nn.LayerNorm(dim)

    def forward(self, inputs, num_slots = None):
        b, n, d = inputs.shape
        n_s = num_slots if num_slots is not None else self.num_slots
        
        mu = self.slots_mu.expand(b, n_s, -1)
        sigma = self.slots_sigma.expand(b, n_s, -1)
        slots = torch.normal(mu, sigma)

        inputs = self.norm_input(inputs)        
        k, v = self.to_k(inputs), self.to_v(inputs)

        for _ in range(self.iters):
            slots_prev = slots

            slots = self.norm_slots(slots)
            q = self.to_q(slots)

            dots = torch.einsum('bid,bjd->bij', q, k) * self.scale
            attn = dots.softmax(dim=1) + self.eps
            attn = attn / attn.sum(dim=-1, keepdim=True)

            updates = torch.einsum('bjd,bij->bid', v, attn)

            slots = self.gru(
                updates.reshape(-1, d),
                slots_prev.reshape(-1, d)
            )

            slots = slots.reshape(b, -1, d)
            slots = slots + self.fc2(F.relu(self.fc1(self.norm_pre_ff(slots))))

        return slots

def build_grid(resolution):
    ranges = [np.linspace(0., 1., num=res) for res in resolution]
    grid = np.meshgrid(*ranges, sparse=False, indexing="ij")
    grid = np.stack(grid, axis=-1)
    grid = np.reshape(grid, [resolution[0], resolution[1], -1])
    grid = np.expand_dims(grid, axis=0)
    grid = grid.astype(np.float32)
    return torch.from_numpy(np.concatenate([grid, 1.0 - grid], axis=-1)).to(device)

"""Adds soft positional embedding with learnable projection."""
class SoftPositionEmbed(nn.Module):
    def __init__(self, hidden_size, resolution):
        """Builds the soft position embedding layer.
        Args:
        hidden_size: Size of input feature dimension.
        resolution: Tuple of integers specifying width and height of grid.
        """
        super().__init__()
        self.embedding = nn.Linear(4, hidden_size, bias=True)
        self.grid = build_grid(resolution)

    def forward(self, inputs):
        grid = self.embedding(self.grid)
        return inputs + grid

class Encoder(nn.Module):
    def __init__(self, resolution, hid_dim, in_channels=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, hid_dim, 5, padding = 2)
        self.conv2 = nn.Conv2d(hid_dim, hid_dim, 5, padding = 2)
        self.conv3 = nn.Conv2d(hid_dim, hid_dim, 5, padding = 2)
        self.conv4 = nn.Conv2d(hid_dim, hid_dim, 5, padding = 2)
        self.encoder_pos = SoftPositionEmbed(hid_dim, resolution)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.conv3(x)
        x = F.relu(x)
        x = self.conv4(x)
        x = F.relu(x)
        x = x.permute(0,2,3,1)
        x = self.encoder_pos(x)
        x = torch.flatten(x, 1, 2)
        return x

class Decoder(nn.Module):
    def __init__(self, hid_dim, resolution):
        super().__init__()
        self.conv1 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv2 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv3 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv4 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv5 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(1, 1), padding=2).to(device)
        self.conv6 = nn.ConvTranspose2d(hid_dim, 2, 3, stride=(1, 1), padding=1)
        self.decoder_initial_size = (8, 8)
        self.decoder_pos = SoftPositionEmbed(hid_dim, self.decoder_initial_size)
        self.resolution = resolution

    def forward(self, x):
        x = self.decoder_pos(x)
        x = x.permute(0,3,1,2)
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
#         x = F.pad(x, (4,4,4,4)) # no longer needed
        x = self.conv3(x)
        x = F.relu(x)
        x = self.conv4(x)
        x = F.relu(x)
        x = self.conv5(x)
        x = F.relu(x)
        x = self.conv6(x)
        x = x[:,:,:self.resolution[0], :self.resolution[1]]
        x = x.permute(0,2,3,1)
        return x

"""Slot Attention-based auto-encoder for object discovery."""
class SlotAttentionAutoEncoder(nn.Module):
    def __init__(self, resolution, num_slots, num_iterations, hid_dim):
        """Builds the Slot Attention-based auto-encoder.
        Args:
        resolution: Tuple of integers specifying width and height of input image.
        num_slots: Number of slots in Slot Attention.
        num_iterations: Number of iterations in Slot Attention.
        """
        super().__init__()
        self.hid_dim = hid_dim
        self.resolution = resolution
        self.num_slots = num_slots
        self.num_iterations = num_iterations

        self.encoder_cnn = Encoder(self.resolution, self.hid_dim)
        self.decoder_cnn = Decoder(self.hid_dim, self.resolution)

        self.fc1 = nn.Linear(hid_dim, hid_dim)
        self.fc2 = nn.Linear(hid_dim, hid_dim)

        self.slot_attention = SlotAttention(
            num_slots=self.num_slots,
            dim=hid_dim,
            iters = self.num_iterations,
            eps = 1e-8, 
            hidden_dim = 128)

    def forward(self, image):
        # `image` has shape: [batch_size, num_channels, width, height].

        # Convolutional encoder with position embedding.
        x = self.encoder_cnn(image)  # CNN Backbone.
        x = nn.LayerNorm(x.shape[1:]).to(device)(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)  # Feedforward network on set.
        # `x` has shape: [batch_size, width*height, input_size].

        # Slot Attention module.
        slots = self.slot_attention(x)
        print(slots.shape)
        # `slots` has shape: [batch_size, num_slots, slot_size].

        # """Broadcast slot features to a 2D grid and collapse slot dimension.""".
        slots = slots.reshape((-1, slots.shape[-1])).unsqueeze(1).unsqueeze(2)
        slots = slots.repeat((1, 8, 8, 1))
        
        # `slots` has shape: [batch_size*num_slots, width_init, height_init, slot_size].
        x = self.decoder_cnn(slots)
        # `x` has shape: [batch_size*num_slots, width, height, num_channels+1].

        # Undo combination of slot and batch dimension; split alpha masks.
        recons, masks = x.reshape(image.shape[0], -1, x.shape[1], x.shape[2], x.shape[3]).split([1,1], dim=-1)
        print("recons:", recons.shape)
        print("masks:", masks.shape)
        # `recons` has shape: [batch_size, num_slots, width, height, num_channels].
        # `masks` has shape: [batch_size, num_slots, width, height, 1].

        # Normalize alpha masks over slots.
        masks = nn.Softmax(dim=1)(masks)
        recon_combined = torch.sum(recons * masks, dim=1)  # Recombine image.
        recon_combined = recon_combined.permute(0,3,1,2)
        # `recon_combined` has shape: [batch_size, width, height, num_channels].
        print(recon_combined.shape)
        return recon_combined, recons, masks, slots

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as dist


class SlotAttention(nn.Module):

    def __init__(self, in_dim, slot_size, num_slots, num_iters, mlp_hdim, 
                                                                epsilon =1e-8, implicit_grads=False):

        super().__init__()

        self.num_slots = num_slots
        self.num_iters = num_iters
        self.slot_size = slot_size
        self.epsilon = epsilon
        self.implicit_grads = implicit_grads

        self.project_q = nn.Linear(slot_size, slot_size, bias=False)
        self.project_k = nn.Linear(in_dim, slot_size, bias=False)
        self.project_v = nn.Linear(in_dim, slot_size, bias=False)

        self.norm_inputs = nn.LayerNorm(in_dim)
        self.norm_slots = nn.LayerNorm(slot_size)
        self.norm_mlp = nn.LayerNorm(slot_size)


        self.gru = nn.GRUCell(slot_size, slot_size)
        self.mlp = nn.Sequential(
            nn.Linear(slot_size, mlp_hdim),
            nn.ReLU(),
            nn.Linear(mlp_hdim, slot_size)
            )

        self.slots_mu = nn.Parameter(nn.init.xavier_uniform_(torch.zeros(1, 1, self.slot_size)))
        self.slots_logsigma = nn.Parameter(nn.init.xavier_uniform_(torch.ones( 1, self.slot_size)))


    def step(self, slots, k, v, batch_size):

        slots_prev = slots
        slots = self.norm_slots(slots)
        q = self.project_q(slots) # shape: [batch_size, num_slots, slot_size]
        scores = (self.slot_size ** -0.5) * torch.matmul(k, q.transpose(2, 1))
        attn = torch.softmax(scores, dim=-1) # shape: [batch_size, num_inputs, num_slots]

        #weighted mean 
        attn = attn + self.epsilon
        attn = attn/ torch.sum(attn, dim=1, keepdim=True) #shape: [batch_size, num_inputs, num_slots]

        updates = torch.matmul(attn.transpose(2, 1), v) #shape: [batch_size, num_slots, slot_size]

        slots = self.gru(updates.reshape(-1, self.slot_size), slots_prev.reshape(-1, self.slot_size))
        slots = slots.reshape(batch_size, self.num_slots, self.slot_size)
        slots = self.norm_mlp(slots)
        #slots = self.mlp(slots)
        slots = slots + self.mlp(slots)

        return slots

    def forward(self, x):

        batch_size, num_inputs, in_dim = x.shape
        x = self.norm_inputs(x)
        k = self.project_k(x) # shape:[batch_size, num_inputs, slot_size]
        v = self.project_v(x) # shape:[batch_size, num_inputs, slot_size]
        print('v', v.shape)

        mu = self.slots_mu.repeat(batch_size, self.num_slots, 1)
        logsigma = self.slots_logsigma.repeat(batch_size, self.num_slots, 1)
        logsigma = F.softplus(logsigma) + 1e-5
        slots_dist = dist.independent.Independent(dist.Normal(loc=mu, scale=logsigma), 1)
        slots = slots_dist.rsample()

        for _ in range(self.num_iters):
            slots = self.step(slots, k, v, batch_size)

        if self.implicit_grads:
            slots = self.step(slots.detach(), k, v, batch_size)

        return slots
    
class SlotAttentionEncoder(nn.Module):

    def __init__(self, 
           resolution, 
           num_slots=2,
           num_iters=3,
           device='cuda',
           in_channels =1, 
           num_hidden = 4,
           hdim = 32,
           slot_size = 64,
           slot_mlp_size = 128,
           decoder_resolution=(28, 56),
           implicit_grads = False):

        super().__init__()

        self.resolution = resolution
        self.in_channels = in_channels
        self.slot_size = slot_size
        self.num_slots = num_slots
        self.decoder_resolution = decoder_resolution

        modules = []
        in_dim = self.in_channels
        for _ in range(num_hidden):
            modules.append(nn.Conv2d(in_dim, hdim, kernel_size=5, stride=1, padding=5//2))
            modules.append(nn.ReLU())
            in_dim = hdim

        self.encoder = nn.Sequential(*modules)
        self.encoder_pos_embed = PositionEmbed(hdim, self.resolution, device)

        self.norm_layer = nn.LayerNorm(hdim)
        self.pre_slot_encode = nn.Sequential(
                                    nn.Linear(hdim, hdim),
                                    nn.ReLU(),
                                    nn.Linear(hdim, hdim)
                                )

        self.slot_attention = SlotAttention(
                                in_dim = hdim,
                                slot_size=self.slot_size,
                                num_slots=self.num_slots,
                                num_iters=num_iters,
                                mlp_hdim =slot_mlp_size,
                                implicit_grads = implicit_grads
                              )

    def forward(self, x):

        batch_size, num_channels, height, width = x.shape

        x = self.encoder(x) 
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.encoder_pos_embed(x) #shape:[batch_size, height, width, hdim]

        x = torch.flatten(x, start_dim=1, end_dim=2) #shape:[batch_size, height*width, hdim]

        x = self.norm_layer(x) 
        x = self.pre_slot_encode(x) 


        print(x.shape)
        slots = self.slot_attention(x) #shape:[batch_size, num_slots, slot_size]

        return slots
    

class SlotAttentionDecoder(nn.Module):

    def __init__(self, 
           resolution=(28, 56), 
           num_slots=2,
           num_iters=3,
           device='cuda',
           in_channels =1, 
           num_hidden = 4,
           hdim = 32,
           slot_size = 64,
           slot_mlp_size = 128,
           decoder_resolution=(28, 56),
           implicit_grads = False):

        super().__init__()

        self.resolution = resolution
        self.in_channels = in_channels
        self.slot_size = slot_size
        self.num_slots = num_slots
        self.decoder_resolution = decoder_resolution
        
        self.decoder_pos_embed = PositionEmbed(slot_size, self.decoder_resolution, device)

        modules = []
        in_dim = slot_size
        for _ in range(num_hidden-1):
            modules.append(nn.ConvTranspose2d(in_dim, hdim, kernel_size=5, stride=1, padding=5//2))
            modules.append(nn.ReLU())
            in_dim = hdim

        modules.append(nn.ConvTranspose2d(32, 2, kernel_size=3, stride=1, padding=3//2))

        self.decoder = nn.Sequential(*modules)

    def forward(self, slots):

        x = slots.reshape(-1,1,1, self.slot_size).repeat(1, *self.decoder_resolution, 1)
        # x has shape: [batch_size*num_slots, decoder_res[0], decoder_res[1], slot_size]

        x = self.decoder_pos_embed(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self.decoder(x) 
        #x has shape:[batch_size*num_slots, num_channels+1, height, width]

        num_channels =1 
        height = 28
        width = 56
        x = x.reshape(-1, self.num_slots, num_channels+1, height, width)

        recon, masks = x.split((1, 1), dim=2)
        masks = F.softmax(masks, dim=1)
        recon_combined = (recon * masks).sum(dim=1)

        return recon_combined, recon

class PositionEmbed(nn.Module):

    def __init__(self, hdim, resolution, device):
        super().__init__()

        self.dense = nn.Linear(4, hdim)
        self.grid = build_grid(resolution).to(device)

    def forward(self, x):

        grid = self.dense(self.grid)
        return x + grid

def build_grid(resolution):

    grid = torch.meshgrid(*[torch.linspace(0.0, 1.0, r) for r in resolution])
    grid = torch.stack(grid, dim=-1)
    grid = torch.reshape(grid, [resolution[0], resolution[1], -1])
    grid = grid.unsqueeze(0)
    grid = torch.cat([grid, 1.0-grid], dim=-1)
    return grid


In [2]:
encoders = SlotAttentionEncoder((28, 56)).to('cuda')
slots = encoders(torch.randn(32, 1, 28, 56, device='cuda'))
decoder = SlotAttentionDecoder((28, 56)).to('cuda')
decoder(slots)[0].shape

slots.shape

/home/shared/venv/lib/python3.11/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


torch.Size([32, 1568, 32])
v torch.Size([32, 1568, 64])


torch.Size([32, 2, 64])

In [ ]:
import numpy as np
from torch import nn
import torch
import torch.nn.functional as F

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

import torch
from torch import nn
from torch.nn import init

class SlotAttention(nn.Module):
    def __init__(self, num_slots, dim, iters = 3, eps = 1e-8, hidden_dim = 128):
        super().__init__()
        self.dim = dim
        self.num_slots = num_slots
        self.iters = iters
        self.eps = eps
        self.scale = dim ** -0.5

        # self.slots_mu = nn.Parameter(torch.randn(1, 1, dim))

        # self.slots_logsigma = nn.Parameter(torch.zeros(1, 1, dim))
        
        self.slots_mu = nn.Parameter(torch.zeros(1, num_slots, dim))
        self.slots_logsigma = nn.Parameter(torch.ones(1, num_slots, dim))

        init.xavier_uniform_(self.slots_logsigma)

        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, dim)
        self.to_v = nn.Linear(dim, dim)

        self.gru = nn.GRUCell(dim, dim)

        hidden_dim = max(dim, hidden_dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(inplace = True),
            nn.Linear(hidden_dim, dim)
        )

        self.norm_input  = nn.LayerNorm(dim)
        self.norm_slots  = nn.LayerNorm(dim)
        self.norm_pre_ff = nn.LayerNorm(dim)

    def forward(self, inputs, num_slots = None):
        b, n, d, device, dtype = *inputs.shape, inputs.device, inputs.dtype
        n_s = num_slots if num_slots is not None else self.num_slots
        
        mu = self.slots_mu.expand(b, n_s, -1)
        sigma = self.slots_logsigma.exp().expand(b, n_s, -1)
        slots = mu + sigma * torch.randn(mu.shape, device = device, dtype = dtype)

        inputs = self.norm_input(inputs)        
        k, v = self.to_k(inputs), self.to_v(inputs)
        
        print(k.shape)

        for _ in range(self.iters):
            slots_prev = slots

            slots = self.norm_slots(slots)
            q = self.to_q(slots)

            dots = torch.einsum('bid,bjd->bij', q, k) * self.scale
            attn = dots.softmax(dim=1) + self.eps

            # attn = attn / attn.sum(dim=-1, keepdim=True)

            updates = torch.einsum('bjd,bij->bid', v, attn)

            slots = self.gru(
                updates.reshape(-1, d),
                slots_prev.reshape(-1, d)
            )

            slots = slots.reshape(b, -1, d)
            slots = slots + self.mlp(self.norm_pre_ff(slots))

        return slots
    


def build_grid(resolution):
    ranges = [np.linspace(0., 1., num=res) for res in resolution]
    grid = np.meshgrid(*ranges, sparse=False, indexing="ij")
    grid = np.stack(grid, axis=-1)
    grid = np.reshape(grid, [resolution[0], resolution[1], -1])
    grid = np.expand_dims(grid, axis=0)
    grid = grid.astype(np.float32)
    return torch.from_numpy(np.concatenate([grid, 1.0 - grid], axis=-1)).to(device)

"""Adds soft positional embedding with learnable projection."""
class SoftPositionEmbed(nn.Module):
    def __init__(self, hidden_size, resolution):
        """Builds the soft position embedding layer.
        Args:
        hidden_size: Size of input feature dimension.
        resolution: Tuple of integers specifying width and height of grid.
        """
        super().__init__()
        self.embedding = nn.Linear(4, hidden_size, bias=True)
        self.grid = build_grid(resolution)

    def forward(self, inputs):
        grid = self.embedding(self.grid)
        return inputs + grid

class Encoder(nn.Module):
    def __init__(self, resolution, hid_dim, in_channels=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, hid_dim, 5, padding = 2, stride=2)
        self.conv2 = nn.Conv2d(hid_dim, hid_dim, 5, padding = 2, stride=2)
        self.conv3 = nn.Conv2d(hid_dim, hid_dim, 5, padding = 2, stride=2)
        self.conv4 = nn.Conv2d(hid_dim, hid_dim, 5, padding = 2, stride=2)
        self.encoder_pos = SoftPositionEmbed(hid_dim, resolution)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.conv3(x)
        x = F.relu(x)
        x = self.conv4(x)
        x = F.relu(x)
        x = x.permute(0,2,3,1)
        x = self.encoder_pos(x)
        x = torch.flatten(x, 1, 2)
        return x

class Decoder(nn.Module):
    def __init__(self, hid_dim, resolution, out_channels=1):
        super().__init__()
        self.conv1 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv2 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv3 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv4 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(2, 2), padding=2, output_padding=1).to(device)
        self.conv5 = nn.ConvTranspose2d(hid_dim, hid_dim, 5, stride=(1, 1), padding=2).to(device)
        self.conv6 = nn.ConvTranspose2d(hid_dim, out_channels+1, 3, stride=(1, 1), padding=1)
        self.decoder_initial_size = (8, 8)
        self.decoder_pos = SoftPositionEmbed(hid_dim, self.decoder_initial_size)
        self.resolution = resolution

    def forward(self, x):
        x = self.decoder_pos(x)
        x = x.permute(0,3,1,2)
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
#         x = F.pad(x, (4,4,4,4)) # no longer needed
        x = self.conv3(x)
        x = F.relu(x)
        x = self.conv4(x)
        x = F.relu(x)
        x = self.conv5(x)
        x = F.relu(x)
        x = self.conv6(x)
        x = x[:,:,:self.resolution[0], :self.resolution[1]]
        x = x.permute(0,2,3,1)
        return x

"""Slot Attention-based auto-encoder for object discovery."""
class SlotAttentionEncoder(nn.Module):
    def __init__(self, resolution=(2, 4), num_slots=3, num_iterations=3, hid_dim=64):
        """Builds the Slot Attention-based auto-encoder.
        Args:
        resolution: Tuple of integers specifying width and height of input image.
        num_slots: Number of slots in Slot Attention.
        num_iterations: Number of iterations in Slot Attention.
        """
        super().__init__()
        self.hid_dim = hid_dim
        self.resolution = resolution
        self.num_slots = num_slots
        self.num_iterations = num_iterations

        self.encoder_cnn = Encoder(self.resolution, self.hid_dim)

        self.fc1 = nn.Linear(hid_dim, hid_dim)
        self.fc2 = nn.Linear(hid_dim, hid_dim)

        self.slot_attention = SlotAttention(
            num_slots=self.num_slots,
            dim=hid_dim,
            iters = self.num_iterations,
            eps = 1e-8, 
            hidden_dim = 128)

    def forward(self, image, num_slots=None):
        # `image` has shape: [batch_size, num_channels, width, height].

        # Convolutional encoder with position embedding.
        image = image.permute(0,3,1,2)
        x = self.encoder_cnn(image)  # CNN Backbone.
        print("after CNN", x.shape)
        x = nn.LayerNorm(x.shape[1:]).to(device)(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)  # Feedforward network on set.
        # `x` has shape: [batch_size, width*height, input_size].

        # Slot Attention module.
        slots = self.slot_attention(x, num_slots=num_slots)
        return slots
    
"""Slot Attention-based auto-encoder for object discovery."""
class SlotAttentionDecoder(nn.Module):
    def __init__(self, resolution=(28, 56), num_slots=3, num_iterations=3, hid_dim=64):
        """Builds the Slot Attention-based auto-encoder.
        Args:
        resolution: Tuple of integers specifying width and height of input image.
        num_slots: Number of slots in Slot Attention.
        num_iterations: Number of iterations in Slot Attention.
        """
        super().__init__()
        self.hid_dim = hid_dim
        self.resolution = resolution
        self.num_slots = num_slots
        self.num_iterations = num_iterations

        self.decoder_cnn = Decoder(self.hid_dim, self.resolution)

    def forward(self, slots, num_slots=None):
        if num_slots == None:
            num_slots = self.num_slots
        # """Broadcast slot features to a 2D grid and collapse slot dimension.""".
        slots = slots.reshape((-1, slots.shape[-1])).unsqueeze(1).unsqueeze(2)
        slots = slots.repeat((1, 8, 8, 1))
        
        # `slots` has shape: [batch_size*num_slots, width_init, height_init, slot_size].
        x = self.decoder_cnn(slots)
        # `x` has shape: [batch_size*num_slots, width, height, num_channels+1].
        out_channel = 1
        # Undo combination of slot and batch dimension; split alpha masks.
        recons, masks = x.reshape(slots.shape[0] // num_slots, -1, x.shape[1], x.shape[2], x.shape[3]).split([out_channel,1], dim=-1)
        

        # `recons` has shape: [batch_size, num_slots, width, height, num_channels].
        # `masks` has shape: [batch_size, num_slots, width, height, 1].

        # Normalize alpha masks over slots.
        masks = nn.Softmax(dim=1)(masks)
        recon_combined = torch.sum(recons * masks, dim=1)  # Recombine image.
        recon_combined = recon_combined.permute(0,3,1,2)
        # `recon_combined` has shape: [batch_size, width, height, num_channels].
        return recon_combined.permute(0,2,3,1), recons

In [27]:
encoders = SlotAttentionEncoder((2, 4)).to('cuda')
slots = encoders(torch.randn(32, 28, 56, 1, device='cuda'))
decoder = SlotAttentionDecoder((28, 56)).to('cuda')
decoder(slots)[0].shape

after CNN torch.Size([32, 8, 64])
torch.Size([32, 8, 64])


torch.Size([32, 28, 56, 1])

In [3]:
pip install slot_attention -i https://pypi.org/simple


Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
from slot_attention import SlotAttention

slot_attn = SlotAttention(
    num_slots = 5,
    dim = 512,
    iters = 3   # iterations of attention, defaults to 3
)

inputs = torch.randn(2, 1024, 512)
slot_attn(inputs) # (2, 5, 512)

tensor([[[-0.5385,  0.6007,  0.0273,  ...,  0.2942, -0.2053, -0.7007],
         [-0.0798,  0.5684, -0.6483,  ..., -0.1080, -0.2526,  0.4884],
         [-0.7144,  0.3109, -0.4062,  ...,  0.0141, -0.0724, -0.0495],
         [ 0.8180,  0.3749,  0.2181,  ...,  0.2012, -0.3183, -0.5841],
         [ 0.1665,  0.7991, -0.6968,  ...,  0.3581,  0.5062, -0.3800]],

        [[-0.0788,  0.6586,  0.5970,  ...,  0.9904,  0.2743, -0.8646],
         [ 0.0812,  0.3924, -0.3644,  ...,  0.2042,  0.1680, -0.3426],
         [ 0.3164,  0.6842, -0.0191,  ...,  0.0221,  0.2564, -0.2721],
         [ 0.1420, -0.0809, -0.6414,  ...,  0.7409, -0.1227,  0.5906],
         [ 0.4238,  0.5950, -0.5046,  ...,  0.9416,  0.0935, -0.6306]]],
       grad_fn=<AddBackward0>)